# EuroSAT dataset import and shared splits

Load EuroSAT using the same `eurosat_splits.csv` and `eurosat_classes.json` files used by `SqueezeNet_EuroSAT.ipynb`. The train, validation, and test assignments remain identical for later model comparison.

## Configuration

Set `data_dir` to an existing EuroSAT directory, or leave it as `None` to download the dataset through KaggleHub. Set `ROOT` explicitly if the kernel starts outside the repository folder.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "eurosat_data.py").is_file():
    raise FileNotFoundError(
        "Set ROOT to the repository folder containing eurosat_data.py"
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

args = SimpleNamespace(
    data_dir=None,  # Or Path("/path/to/EuroSAT")
    splits=ROOT / "eurosat_splits.csv",
    classes=ROOT / "eurosat_classes.json",
    
    batch_size=64,
    num_workers=0,
)

## Imports and shared manifest

`load_splits` validates the class order, labels, paths, and exact split counts before images are loaded.

In [ ]:
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from eurosat_data import load_splits, validate_images

rows, class_names = load_splits(args.splits, args.classes)
manifest = pd.DataFrame(rows)
NUM_CLASSES = len(class_names)

print(manifest["split"].value_counts())
print("Classes:", class_names)

## Locate and validate the EuroSAT images

The same KaggleHub dataset source and `EuroSAT` directory layout are used as in the SqueezeNet notebook.

In [ ]:
if args.data_dir is None:
    import kagglehub
    dataset_path = kagglehub.dataset_download("apollo2506/eurosat-dataset")
    DATA_DIR = Path(dataset_path) / "EuroSAT"
else:
    DATA_DIR = Path(args.data_dir)

validate_images(rows, DATA_DIR)
print("Dataset directory:", DATA_DIR)
print("All manifest images found.")

## Create train, validation, and test datasets

Use the same 64 x 64 RGB preprocessing and ImageNet normalization as SqueezeNet. No split is reshuffled or regenerated.

In [ ]:
class EuroSATDataset(Dataset):
    def __init__(self, manifest, split, root, transform):
        self.frame = manifest[manifest["split"] == split].reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.root / row["relative_path"]) as image:
            image = self.transform(image.convert("RGB"))
        return image, int(row["label"])


image_transform = transforms.Compose([
    transforms.Resize(
        (64, 64),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = EuroSATDataset(manifest, "train", DATA_DIR, image_transform)
val_dataset = EuroSATDataset(manifest, "val", DATA_DIR, image_transform)
test_dataset = EuroSATDataset(manifest, "test", DATA_DIR, image_transform)

generator = torch.Generator().manual_seed(42)
loader_options = {
    "batch_size": args.batch_size,
    "num_workers": args.num_workers,
    "pin_memory": torch.cuda.is_available(),
}
train_loader = DataLoader(
    train_dataset, shuffle=True, generator=generator, **loader_options
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

## Model A: Standard CNN

Three blocks of Conv 3x3 → BatchNorm → ReLU → MaxPool 2x2 (32, 64, 128 filters), then global average pooling, Dense 64 + ReLU + Dropout(0.3), and a Dense 10 output.

- **Parameters:** 102,826 total (102,378 trainable + 448 BatchNorm running statistics), about 0.41 MB in float32.
- **`bias=False`** in the convolutions, because BatchNorm already has its own shift.
- **ReLU** needs only a comparison with zero (no exponential), so it is cheap on edge hardware.
- **Global average pooling** keeps the dense head small.
- **Output is raw logits.** `CrossEntropyLoss` applies softmax internally.

In [ ]:
import torch
from torch import nn
from collections import OrderedDict


def create_model_a(input_shape=(64, 64, 3), num_classes=10):
    in_channels = input_shape[2]          # (H, W, C) -> C

    model = nn.Sequential(OrderedDict([
        # ---------------- Block 1 ----------------
        # bias=False, since we use batch normalization
        ("conv1", nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False)),
        ("bn1", nn.BatchNorm2d(32)),
        ("relu1", nn.ReLU()),
        ("pool1", nn.MaxPool2d(kernel_size=2)),          # -> (32, 32, 32)

        # ---------------- Block 2 ----------------
        ("conv2", nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)),
        ("bn2", nn.BatchNorm2d(64)),
        ("relu2", nn.ReLU()),
        ("pool2", nn.MaxPool2d(kernel_size=2)),          # -> (64, 16, 16)

        # ---------------- Block 3 ----------------
        ("conv3", nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)),
        ("bn3", nn.BatchNorm2d(128)),
        ("relu3", nn.ReLU()),
        ("pool3", nn.MaxPool2d(kernel_size=2)),          # -> (128, 8, 8)

        # ---------------- Classifier Head ----------------
        # Global average pooling aggregates spatial dimensions into a compact 128-d vector
        ("gap", nn.AdaptiveAvgPool2d(1)),                # -> (128, 1, 1)
        ("flatten", nn.Flatten()),                       # -> (128)
        ("dense1", nn.Linear(128, 64)),
        ("relu_fc", nn.ReLU()),
        ("dropout", nn.Dropout(0.3)),
        ("predictions", nn.Linear(64, num_classes)),     # logits (no softmax)
    ]))

    # Same defaults as Keras: Glorot-uniform weights, zero biases
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    return model


# Instantiate and verify parameters
model_a = create_model_a()
print(model_a)

trainable = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
non_trainable = sum(b.numel() for n, b in model_a.named_buffers()
                    if "num_batches_tracked" not in n)
print(f"Trainable parameters:     {trainable:,}")       # 102,378
print(f"Non-trainable parameters: {non_trainable:,}")   # 448 (BatchNorm running stats)
print(f"Total parameters:         {trainable + non_trainable:,}")  # 102,826

# Shape check (PyTorch input is channels-first: batch, 3, 64, 64)
print(model_a(torch.randn(2, 3, 64, 64)).shape)         # torch.Size([2, 10])

### Model B

In [ ]:
class DepthwiseSeparableBlock(torch.nn.Module):
    """Depthwise 3x3 convolution followed by pointwise 1x1 convolution."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv2d(
                in_channels=in_channels,
                out_channels=in_channels,
                kernel_size=3,
                padding=1,
                groups=in_channels,
                bias=False,
            ),
            torch.nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
                bias=False,
            ),
            torch.nn.BatchNorm2d(out_channels),
            torch.nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ModelBLightweightCNN(torch.nn.Module):
    def __init__(self, input_shape=(64, 64, 3), num_classes=10):
        super().__init__()
        height, width, input_channels = input_shape
        if (height, width) != (64, 64) or input_channels != 3:
            raise ValueError("input_shape must be (64, 64, 3)")

        self.features = torch.nn.Sequential(
            DepthwiseSeparableBlock(input_channels, 32),
            torch.nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableBlock(32, 64),
            torch.nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableBlock(64, 128),
            torch.nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(p=0.3),
            torch.nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def create_model_b(input_shape=(64, 64, 3), num_classes=10):
    return ModelBLightweightCNN(input_shape=input_shape, num_classes=num_classes)


# Instantiate and verify parameters.
model_b = create_model_b(num_classes=NUM_CLASSES)
print(model_b)

# Parameter threshold constraint validation.
total_params = sum(parameter.numel() for parameter in model_b.parameters())
assert total_params < 100_000, (
    f"Parameter count {total_params} exceeds the 100k limit!"
)
print(
    f"\n[PASS] Model B Parameter Count: {total_params:,} "
    "(Strictly under 100,000 threshold)"
)